# Building a RAG System with LangChain, LangGraph & OpenAI

This notebook demonstrates how to build a complete Retrieval-Augmented Generation (RAG) system using:
- **LangChain**: For document processing and LLM orchestration
- **LangGraph**: For workflow management
- **FAISS**: For in-memory vector storage and retrieval
- **OpenAI**: For embeddings and text generation

## Architecture Overview

1. **Document Loading**: Read documents from a folder
2. **Chunking**: Split documents into manageable pieces
3. **Vectorization**: Convert chunks to embeddings using OpenAI
4. **Storage**: Store vectors in-memory using FAISS
5. **Retrieval**: Find relevant chunks for user questions
6. **Prompt Creation**: Build system prompt with retrieved context
7. **Generation**: Generate answers using OpenAI
8. **Workflow**: Orchestrate everything with LangGraph


## Step 1: Setup and Dependencies

First, let's install and import all necessary packages.


# Requirements
langchain  
langchain-openai  
langchain-community  

python-dotenv 

faiss-cpu  

In [ ]:
%pip install langchain langchain-openai langchain-community python-dotenv faiss-cpu -q

In [ ]:
import glob
import json
import os
from typing import Any

from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [ ]:
# Load environment variables
load_dotenv()

print("✅ All packages imported successfully!")

In [ ]:
# Configuration - Set your API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Set in .env file or replace with your key

# Configuration parameters
DOCUMENTS_FOLDER = "./documents"  # Folder containing your documents (supports nested folders)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

print("✅ Configuration loaded!")
print(f"📁 Documents folder: {DOCUMENTS_FOLDER}")
print(f"📏 Chunk size: {CHUNK_SIZE} with {CHUNK_OVERLAP} overlap")
print(f"🔤 Embedding model: {EMBEDDING_MODEL}")
print(f"🤖 LLM model: {LLM_MODEL}")


## Step 2: Document Loading

Load documents from a specified folder.


In [76]:
def load_documents_from_folder(folder_path: str) -> list[Document]:
    """Load all txt documents from a single folder."""
    # Find all txt files in the folder
    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))
    documents = []

    for file_path in txt_files:
        # Load the document
        loader = TextLoader(file_path)
        docs = loader.load()

        # prepare metadata
        filename = os.path.basename(file_path)
        company = filename.split("_")[0].lower()

        for doc in docs:
            doc.metadata.update(
                {
                    "source_file": filename,
                    "file_type": ".txt",
                    "full_path": file_path,
                    "company": company,
                }
            )

        documents.extend(docs)

    return documents


# Load documents from the simplified folder structure
documents = load_documents_from_folder(DOCUMENTS_FOLDER)

# Print comprehensive statistics and examples
if documents:
    # Company statistics
    companies = {}
    for doc in documents:
        company = doc.metadata.get("company", "unknown")
        companies[company] = companies.get(company, 0) + 1

    print(f"\n📚 Total documents loaded: {len(documents)}")
    print("\n📊 Documents loaded by company:")
    for company, count in sorted(companies.items()):
        print(f"  • {company}: {count} documents")

    # Show some example metadata
    print("\n🔍 Example document metadata:")
    print("=" * 50)
    for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"Document {i + 1}:")
        print(f"  • Source: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"  • Company: {doc.metadata.get('company', 'Unknown')}")
        print(f"  • Content preview: {doc.page_content[:100]}...")
        print("-" * 30)
else:
    print("❌ No documents were loaded!")

documents[0]



📚 Total documents loaded: 15

📊 Documents loaded by company:
  • amazon: 3 documents
  • apple: 3 documents
  • google: 3 documents
  • microsoft: 3 documents
  • tesla: 3 documents

🔍 Example document metadata:
Document 1:
  • Source: amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt
  • Company: amazon
  • Content preview: COMPANY: AMAZON
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:26
========================================...
------------------------------
Document 2:
  • Source: google_yes_google_meet_is_down_20250914_151020.txt
  • Company: google
  • Content preview: COMPANY: GOOGLE
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:20
========================================...
------------------------------


Document(metadata={'source': './documents/amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt', 'source_file': 'amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt', 'file_type': '.txt', 'full_path': './documents/amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt', 'company': 'amazon'}, page_content='COMPANY: AMAZON\nARTICLE: 3/3\nGENERATED: 2025-09-14 15:10:26\n================================================================================\n\nTITLE: Amazon’s Lens Live AI shops for anything you can see\nSOURCE: The Verge\nAUTHOR: Emma Roth\nPUBLISHED: 2025-09-02T21:01:13Z\nURL: https://www.theverge.com/news/769585/amazon-lens-live-ai-real-time-shopping\nDESCRIPTION: Amazon will now let you shop for products by pointing your camera at them. On Thursday, the company announced Lens Live, a new feature that uses your camera to scan things in the environment around you, while surfacing matching product listings. This feature,…\nEXTRACT

## Step 3: Document Chunking

Split documents into smaller, manageable chunks that can be effectively vectorized and retrieved.


In [77]:
def chunk_documents(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 200
) -> list[Document]:
    """
    Split documents into smaller chunks for better retrieval.
    """

    # Initialize the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],  # Try to split on paragraphs, then lines, then words
    )

    # Split documents
    chunked_docs = text_splitter.split_documents(documents)

    # Add chunk information to metadata while preserving original metadata
    for i, chunk in enumerate(chunked_docs):
        # Preserve all original metadata and add chunk-specific information
        chunk.metadata.update(
            {
                "chunk_id": i,
                "chunk_size": len(chunk.page_content),
                "total_chunks": len(chunked_docs),
            }
        )

    return chunked_docs


In [78]:
# Chunk the loaded documents
chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

# Print comprehensive chunk statistics and examples
if chunks:
    print(f"\n📄 Split {len(documents)} documents into {len(chunks)} chunks")

    # Chunk size statistics
    chunk_sizes = [len(chunk.page_content) for chunk in chunks]
    print(
        f"📊 Chunk size stats: min={min(chunk_sizes)}, max={max(chunk_sizes)}, avg={sum(chunk_sizes) // len(chunk_sizes)}"
    )

    # Company distribution in chunks
    company_chunks = {}
    for chunk in chunks:
        company = chunk.metadata.get("company", "Unknown")
        company_chunks[company] = company_chunks.get(company, 0) + 1

    print(f"📊 Chunks by company: {dict(sorted(company_chunks.items()))}")

    # Display first chunk as example
    print("\n🔍 Example chunk:")
    print("=" * 50)
    print(f"Content: {chunks[0].page_content[:200]}...")
    print(f"Company: {chunks[0].metadata.get('company', 'Unknown')}")
    print(f"Source File: {chunks[0].metadata.get('source_file', 'Unknown')}")
    print(f"Chunk ID: {chunks[0].metadata.get('chunk_id', 'Unknown')}")

    print(f"Full Metadata: {json.dumps(chunks[0].metadata, indent=4)}")
    print("=" * 50)
else:
    print("❌ No chunks were created!")



📄 Split 15 documents into 80 chunks
📊 Chunk size stats: min=145, max=997, avg=794
📊 Chunks by company: {'amazon': 9, 'apple': 17, 'google': 20, 'microsoft': 20, 'tesla': 14}

🔍 Example chunk:
Content: COMPANY: AMAZON
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:26

TITLE: Amazon’s Lens Live AI shops for anything you can se...
Company: amazon
Source File: amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt
Chunk ID: 0
Full Metadata: {
    "source": "./documents/amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt",
    "source_file": "amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt",
    "file_type": ".txt",
    "full_path": "./documents/amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt",
    "company": "amazon",
    "chunk_id": 0,
    "chunk_size": 972,
    "total_chunks": 80
}


## Step 4: Vectorization with OpenAI Embeddings

Convert text chunks into vector embeddings using OpenAI's embedding models.


In [53]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

# Test embeddings with a sample text
sample_text = "Machine learning is a subset of artificial intelligence."
sample_embedding = embeddings.embed_query(sample_text)

print("✅ Embeddings initialized!")
print(f"📊 Embedding model: {EMBEDDING_MODEL}")
print(f"📏 Embedding dimension: {len(sample_embedding)}")
print(f"🔍 Sample embedding (first 5 values): {sample_embedding[:5]}")


✅ Embeddings initialized!
📊 Embedding model: text-embedding-3-small
📏 Embedding dimension: 1536
🔍 Sample embedding (first 5 values): [-0.021235132589936256, -0.05318146198987961, -0.01300511509180069, -0.028837835416197777, 0.055503468960523605]


## Step 5: In-Memory Vector Storage with FAISS

Set up FAISS in-memory vector database and store the vectorized chunks for efficient similarity search.


In [54]:
def create_embeddings_for_chunks(
    chunks: list[Document], embeddings_model: OpenAIEmbeddings
) -> list[list[float]]:
    """
    Create embeddings for all document chunks.
    """
    print(f"🔄 Creating embeddings for {len(chunks)} chunks...")

    # Extract text content from chunks
    texts = [chunk.page_content for chunk in chunks]

    # Create embeddings in batches for efficiency
    embeddings_list = embeddings_model.embed_documents(texts)

    print(f"✅ Created {len(embeddings_list)} embeddings")
    return embeddings_list


# Create embeddings for our chunks
chunk_embeddings = create_embeddings_for_chunks(chunks, embeddings)


🔄 Creating embeddings for 80 chunks...
✅ Created 80 embeddings


In [55]:
def create_faiss_vector_store(chunks: list[Document], embeddings_model) -> FAISS:
    """
    Create FAISS vector store from document chunks.

    Args:
        chunks: List of document chunks
        embeddings_model: OpenAI embeddings model

    Returns:
        FAISS vector store
    """
    print(f"🔄 Creating FAISS vector store with {len(chunks)} documents...")

    # Extract texts and metadatas
    texts = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    # Create FAISS vector store directly from texts
    # This will automatically generate embeddings for each text
    vector_store = FAISS.from_texts(texts=texts, embedding=embeddings_model, metadatas=metadatas)

    print("✅ Successfully created FAISS vector store!")
    print("📊 Vector store stats:")
    print(f"  • Total vectors: {vector_store.index.ntotal}")
    print(f"  • Vector dimension: {vector_store.index.d}")

    return vector_store


# Create FAISS vector store
vector_store = create_faiss_vector_store(chunks, embeddings)


🔄 Creating FAISS vector store with 80 documents...
✅ Successfully created FAISS vector store!
📊 Vector store stats:
  • Total vectors: 80
  • Vector dimension: 1536


In [56]:
# Test retrieval
test_query = "What is the latest news about Amazon?"

# Test retrieval with scores
print(f"\n{'_' * 50}\n🔍 Testing retrieval with similarity scores...")
similar_docs_with_scores = vector_store.similarity_search_with_score(test_query, k=2)

print(f"Query: {test_query}")
for i, (doc, score) in enumerate(similar_docs_with_scores):
    print(f"\n--- Found Document {i + 1} (Score: {score:.4f}) ---")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Source: {doc.metadata.get('source_file', 'Unknown')}")



__________________________________________________
🔍 Testing retrieval with similarity scores...
Query: What is the latest news about Amazon?

--- Found Document 1 (Score: 0.9529) ---
Content: COMPANY: AMAZON
ARTICLE: 1/3
GENERATED: 2025-09-14 15:10:26

TITLE: A...
Source: amazon_next_tablet_might_run_android_20250914_151026.txt

--- Found Document 2 (Score: 0.9842) ---
Content: is a news writer who covers the streaming wars, consumer tech, crypto, social media, and much more. Previously, she was a writer and editor at MUO.

P...
Source: amazon_next_tablet_might_run_android_20250914_151026.txt


## Step 6: Retrieval System

Build a sophisticated retrieval system that finds the most relevant document chunks for a given query.


In [ ]:
class RAGRetriever:
    """
    Advanced retrieval system for RAG pipeline.
    Always returns documents with similarity scores for transparency.
    """

    def __init__(self, vector_store: FAISS, top_k: int = 5):
        self.vector_store = vector_store
        self.top_k = top_k

    def retrieve_documents(self, query: str) -> list[tuple]:
        """Retrieve relevant documents with similarity scores."""
        docs_with_scores = self.vector_store.similarity_search_with_score(query, k=self.top_k)
        return docs_with_scores

    def get_documents_only(self, query: str,) -> list[Document]:
        """Get just the documents without scores (for context generation)."""
        docs_with_scores = self.retrieve_documents(query, self.top_k)
        return [doc for doc, score in docs_with_scores]

    def get_context_string(self, query: str) -> str:
        """Get formatted context string from retrieved documents."""
        docs = self.get_documents_only(query, self.top_k)

        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source_file", "Unknown")
            company = doc.metadata.get("company", "")
            company_prefix = f"({company}) " if company and company != "root" else ""
            content = doc.page_content.strip()
            context_parts.append(f"[Source {i}: {company_prefix}{source}]\\n{content}")

        return "\n\n".join(context_parts)


In [81]:
# Initialize retriever
retriever = RAGRetriever(vector_store, top_k=3)

# Test the retrieval system
print("🔍 Testing Retrieval System")
print("=" * 50)

test_queries = [
    "What are the latest news about Amazon?",
    "I want to learn Machine Learning",
]

for query in test_queries:
    print(f"\n🤔 Query: {query}")

    # Get documents with scores (now the default behavior)
    docs_with_scores = retriever.retrieve_documents(query)

    print(f"📄 Retrieved {len(docs_with_scores)} documents:")
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        print(f"\n--- Document {i} (Score: {score:.4f}) ---")
        print(f"Source: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"Content: {doc.page_content[:200]}...")

print("\n" + "=" * 50)


🔍 Testing Retrieval System

🤔 Query: What are the latest news about Amazon?
📄 Retrieved 3 documents:

--- Document 1 (Score: 0.9399) ---
Source: amazon_next_tablet_might_run_android_20250914_151026.txt
Content: COMPANY: AMAZON
ARTICLE: 1/3
GENERATED: 2025-09-14 15:10:26

TITLE: Amazon’s next tablet might run Android
SOURCE: The ...

--- Document 2 (Score: 0.9737) ---
Source: amazon_next_tablet_might_run_android_20250914_151026.txt
Content: is a news writer who covers the streaming wars, consumer tech, crypto, social media, and much more. Previously, she was a writer and editor at MUO.

Posts from this author will be added to your daily ...

--- Document 3 (Score: 0.9901) ---
Source: amazon_lens_live_ai_shops_for_anything_you_can_se_20250914_151026.txt
Content: COMPANY: AMAZON
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:26

TITLE: Amazon’s Lens Live AI shops for anything you can se...

🤔 Query: I want to learn Machine Learning
📄 Retrieved 3 documents:

--- Document 1 (Score: 1.5136) ---
So

Typical Score Ranges:
* 0.0 = Perfect match (identical vectors)
* 0.0 - 1.0 = Very high similarity
* 1.0 - 2.0 = Good similarity
* 2.0+ = Lower similarity

## Step 7: System Prompt Generation

Create dynamic system prompts using the retrieved context to guide the LLM's response.


In [ ]:
SYSTEM_TEMPLATE = """
You are a helpful AI assistant that answers questions based on the provided context.

Your task is to:
1. Analyze the provided context carefully
2. Answer the user's question using information from the context
3. Be accurate and cite your sources when possible
4. If the context doesn't contain enough information, say so clearly
5. If you're unsure about something, acknowledge the uncertainty

Context Information:
{context}
"""

HUMAN_TEMPLATE = """
Question: {question}
Please provide a detailed answer based on the context above.
"""


def generate_rag_prompt(question: str, retriever: RAGRetriever) -> ChatPromptTemplate:
    """
    Generate a complete RAG prompt by retrieving documents and formatting context.

    Args:
        question: The user's question
        retriever: RAGRetriever instance to get documents and context

    Returns:
        ChatPromptTemplate with context and question formatted
    """
    # Create the prompt template
    prompt_template = ChatPromptTemplate.from_messages(
        [("system", SYSTEM_TEMPLATE), ("human", HUMAN_TEMPLATE)]
    )

    # Use retriever to get formatted context directly
    context = retriever.get_context_string(question, retriever.top_k)
    return prompt_template.partial(context=context)


In [82]:
# Test prompt generation
print("📝 Testing Prompt Generation")
print("=" * 50)

test_question = "How long will the latest apple watch battery last?"

# Generate the prompt using the new function (much simpler!)
prompt = generate_rag_prompt(test_question, retriever)

# Display the formatted prompt
formatted_messages = prompt.format_messages(question=test_question)

print("🤖 Generated System Prompt:")
print("-" * 30)
for message in formatted_messages:
    print(f"**{message.type.upper()}:**")
    print(message.content)
    print("-" * 30)

print("✅ Prompt generation complete!")


📝 Testing Prompt Generation
🤖 Generated System Prompt:
------------------------------
**SYSTEM:**

You are a helpful AI assistant that answers questions based on the provided context. 

Your task is to:
1. Analyze the provided context carefully
2. Answer the user's question using information from the context
3. Be accurate and cite your sources when possible
4. If the context doesn't contain enough information, say so clearly
5. If you're unsure about something, acknowledge the uncertainty

Context Information:
[Source 1: (apple) apple_has_announced_the_apple_watch_series_11_20250914_151007.txt]\nPrevious Next







1 / 5 The color options for the new Apple Watch Series 11. Screenshot: Apple

Apple says the Series 11 will get “up to 24 hours” of battery life. The aluminum version will come in jet black, space gray, rose gold, and silver; the polished titanium one will come in natural, gold, and slate. The watch also comes with Ion-X glass, which Apple says has a ceramic coating bonded

## Step 8: Answer Generation

Generate comprehensive answers using OpenAI's language model with the retrieved context.


In [ ]:
def generate_answer(question: str, llm: ChatOpenAI, retriever: RAGRetriever) -> dict[str, Any]:
    """
    Generate an answer using the provided LLM and retriever.
    """
    # Retrieve documents with scores for source information
    prompt = generate_rag_prompt(question, retriever)
    formatted_messages = prompt.format_messages(question=question)
    response = llm.invoke(formatted_messages)

    return {
        "question": question,
        "answer": response.content,
        "sources": [
            {
                "file": doc.metadata.get("source_file", "Unknown"),
                "company": doc.metadata.get("company", "Unknown"),
                "similarity_score": float(score),
            }
            for doc, score in docs_with_scores
        ],
    }

In [83]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0.1,  # Low temperature for more consistent, factual responses
    openai_api_key=OPENAI_API_KEY,
)


# Test the answer generator
print("🤖 Testing RAG Answer Generation")
print("=" * 60)

test_questions = [
    "What are the latest news about Apple?",
    "How do I learn Machine Learning?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n🤔 Question {i}: {question}")
    print("-" * 60)

    # Generate answer (always with scores now)
    result = generate_answer(question, llm, retriever)

    print("🤖 **Answer:**")
    print(result["answer"])


🤖 Testing RAG Answer Generation

🤔 Question 1: What are the latest news about Apple?
------------------------------------------------------------
🤖 **Answer:**
The latest news about Apple includes the announcement of several new products during their recent event. Here are the key highlights:

1. **iPhone 17 and iPhone Air**: Apple has launched the iPhone 17 and iPhone Air, although specific details about their features were not provided in the context.

2. **Apple Watch Series 11**: The Apple Watch Series 11 has been announced as the slimmest version to date. It includes significant new features such as:
   - **5G Cellular Connectivity**: This is the first Apple Watch to support 5G, which is expected to enhance connectivity, especially in areas with weak signals.
   - **Redesigned Cellular Antenna**: This new design aims to improve coverage.
   - **Live Translation Capabilities**: Similar to the upcoming AirPods Pro 3, the Series 11 will also feature live translation.

3. **AirPods Pr

In [84]:
NO_RAG_SYSTEM_PROMPT = """
You are a helpful AI assistant that answers questions based on your training data knowledge. 

Your task is to:
1. Provide accurate information based on your knowledge
2. Be honest about the limitations of your knowledge
3. Acknowledge when information might be outdated
4. Maintain a helpful and professional tone
5. If you're unsure about recent developments, clearly state this

Please provide comprehensive answers while being transparent about the source of your information.
"""
for question in test_questions:
    no_rag_human_prompt = f"""Question: {question}
Please provide a detailed answer based on your knowledge.
"""
    no_rag_prompt = ChatPromptTemplate.from_messages(
        [("system", NO_RAG_SYSTEM_PROMPT), ("human", no_rag_human_prompt)]
    )
    no_rag_messages = no_rag_prompt.format_messages(question=question)
    no_rag_response = llm.invoke(no_rag_messages)
    print(f"{'-' * 40}\n🤖 **Answer without RAG (Direct LLM):**\n{no_rag_response.content}")


----------------------------------------
🤖 **Answer without RAG (Direct LLM):**
As of my last knowledge update in October 2023, I cannot provide real-time news or updates. However, I can summarize some of the significant developments related to Apple up to that point.

1. **iPhone 15 Launch**: Apple launched the iPhone 15 series in September 2023, which included the iPhone 15, iPhone 15 Plus, iPhone 15 Pro, and iPhone 15 Pro Max. The new models featured improvements in camera technology, battery life, and processing power, with the Pro models incorporating the A17 Pro chip, which offered enhanced graphics performance.

2. **USB-C Transition**: With the iPhone 15 series, Apple transitioned from the Lightning connector to USB-C, aligning with new regulations in the European Union aimed at standardizing charging ports for electronic devices.

3. **Apple Watch Series 9 and Ultra 2**: Alongside the iPhone 15, Apple also introduced the Apple Watch Series 9 and the second-generation Apple Wat

## Next Steps & Production Considerations

### 💡 Why FAISS for This Tutorial?

**Advantages of In-Memory Storage:**
- ✅ **No API Keys Required**: No external service setup needed
- ✅ **Instant Setup**: Works immediately without configuration
- ✅ **Cost-Free**: Perfect for learning and development
- ✅ **Fast Performance**: In-memory operations are very fast
- ✅ **Offline Capable**: Works without internet connection
- ✅ **Persistence Option**: Can save/load indexes to/from disk

**When to Consider External Vector DBs:**
- 🔄 **Large Scale**: Millions of documents
- 🌐 **Multi-User**: Production applications
- 💾 **Persistence**: Long-term storage requirements
- 🔒 **Enterprise Features**: Advanced security, monitoring

### 🔧 Enhancements for Production

1. **Document Processing**
   - Add support for more file types (DOCX, HTML, CSV)
   - Implement document preprocessing (cleaning, normalization)
   - Add document metadata enrichment

2. **Chunking Strategies**
   - Experiment with different chunk sizes and overlaps
   - Implement semantic chunking based on content structure
   - Add chunk quality scoring

3. **Retrieval Improvements**
   - Implement hybrid search (vector + keyword)
   - Add query expansion and rewriting
   - Use re-ranking models for better relevance

4. **Vector Storage Scaling**
   - Move from FAISS to persistent vector databases (Pinecone, Weaviate, Qdrant)
   - Implement efficient batch updates
   - Add vector database monitoring and backup strategies

5. **LLM Integration**
   - Add response streaming for better UX
   - Implement response caching
   - Add support for multiple LLM providers

6. **Monitoring & Evaluation**
   - Add retrieval quality metrics
   - Implement answer quality scoring
   - Set up performance monitoring

### 🚀 Deployment Options

- **Local**: Run on local machine or server
- **Cloud**: Deploy on AWS, GCP, or Azure
- **Containerized**: Use Docker for consistent deployment
- **API**: Wrap in FastAPI or Flask for web service
- **Streamlit**: Create interactive web interface

### 📚 Additional Resources

- [LangChain Documentation](https://python.langchain.com/)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [FAISS Documentation](https://faiss.ai/)
- [OpenAI API Documentation](https://platform.openai.com/docs)
- [Vector Database Comparison](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/vectorstores)

---

**🎥 Perfect for your YouTube video! This notebook covers all the essential steps to build a production-ready RAG system.**


In [ ]:
# RAG vs Non-RAG Comparison
print("🆚 RAG vs Non-RAG Comparison")
print("=" * 80)
print("Let's compare answers with and without RAG to see the difference!")
print("=" * 80)

# Define no-RAG system and human prompts
no_rag_system_prompt = """You are a helpful AI assistant that answers questions based on your training data knowledge. 

Your task is to:
1. Provide accurate information based on your knowledge
2. Be honest about the limitations of your knowledge
3. Acknowledge when information might be outdated
4. Maintain a helpful and professional tone
5. If you're unsure about recent developments, clearly state this

Please provide comprehensive answers while being transparent about the source of your information."""

# Test questions for comparison
comparison_questions = [
    "What are the latest news about Apple?",
    "Tell me about recent developments at Tesla",
    "What's happening with Google's AI products?",
]

for i, question in enumerate(comparison_questions, 1):
    print(f"\n🔍 COMPARISON {i}/3")
    print(f"❓ Question: {question}")
    print("=" * 80)

    # 1. Generate answer WITHOUT RAG (direct LLM)
    print("🤖 **ANSWER WITHOUT RAG (Direct LLM):**")
    print("-" * 40)

    # Create no-RAG human prompt
    no_rag_human_prompt = f"""Question: {question}

Please provide a detailed answer based on your knowledge. If this involves recent developments or current events, please acknowledge any limitations in your knowledge."""

    # Create messages for the LLM
    messages = [
        {"role": "system", "content": no_rag_system_prompt},
        {"role": "user", "content": no_rag_human_prompt},
    ]

    direct_response = llm.invoke(messages)
    print(direct_response.content)
    print("📊 Sources: None (LLM knowledge only)")

    print("\n" + "-" * 80 + "\n")

    # 2. Generate answer WITH RAG
    print("🤖 **ANSWER WITH RAG (Context-Enhanced):**")
    print("-" * 40)

    # Use the existing answer generator (check which one is available)
    try:
        rag_result = answer_generator.generate_answer(question, top_k=2)
    except NameError:
        print("❌ answer_generator not found. Please run the previous cells first.")
        continue

    print(rag_result["answer"])

    print(f"\n📚 **RAG Sources Used ({rag_result['num_sources']}):**")
    for j, source in enumerate(rag_result["sources"], 1):
        company_info = (
            f" ({source['company']})"
            if source.get("company") and source["company"] != "Unknown"
            else ""
        )
        score_info = (
            f" (Score: {source['similarity_score']:.4f})" if "similarity_score" in source else ""
        )
        print(f"  {j}. {source['file']}{company_info}{score_info}")

    print("\n" + "=" * 80)

print("\n🎯 **Key Differences:**")
print("✅ RAG Answers: Based on specific, recent documents with source attribution")
print("✅ RAG Answers: Include similarity scores for transparency")
print("✅ RAG Answers: Can provide company-specific, up-to-date information")
print("❌ Non-RAG Answers: Limited to LLM's training data (potentially outdated)")
print("❌ Non-RAG Answers: No source attribution or verification")
print("❌ Non-RAG Answers: May hallucinate or provide generic responses")

print(
    "\n🚀 This demonstrates the power of RAG for domain-specific, up-to-date information retrieval!"
)
